# P1 — ¿Qué es el over-squashing?

Una GNN de paso de mensajes actualiza cada nodo mezclando a sus vecinos, un salto a la vez. Cuando muchos
caminos confluyen en un nodo, todos sus mensajes se aplastan en un vector de ancho fijo y se pierde señal. Lo
hacemos preciso en un grafo **cuello de botella** que puedes construir con `aiq-quivers`.

## 1. El paso de mensajes avanza un salto por capa

Para alcanzar un nodo a `g` saltos necesitas `g` capas — y todo lo del camino se vuelve a apretar en cada paso.

<img src="../figs-theory/es/E1a_message_one_hop.svg" width="760"/>

## 2. El cuello de botella y la fórmula K·M^d

<img src="../figs-theory/es/T2_bottleneck_formula.svg" width="760"/>

In [ ]:
# Construye el cuello de botella con el helper de aiq-quivers y míralo.
import torch
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash import viz
K, M = 4, 3
data = make_bottleneck_graph(K, M, depth=2, generator=torch.Generator().manual_seed(0))
viz.draw_bottleneck(data, K, M, title='K fuentes -> d capas de M -> objetivo')

## 3. Un vector de ancho fijo se desborda

El conteo de recorridos hacia el objetivo crece como `K·M^d`, pero el vector del objetivo no cambia de tamaño.

<img src="../figs-theory/es/E1b_vector_overflow.svg" width="760"/>

In [ ]:
# Cuenta los mensajes forzados al objetivo, para profundidad creciente (vía operadores de walk).
from oversquash.ideal_bridge import walk_operators
for d in [1, 2, 3, 4]:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, _ = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    t = int(dd.root_mask.nonzero()[0])
    print(f'profundidad {d}: {int(raw[d][:, t].sum()):4d} mensajes apretados en un vector  (~ K*M^d)')

## 4. El conteo de recorridos explota con la distancia

<img src="../figs-theory/es/E1c_explosion.svg" width="760"/>

**Siguiente (P2):** para razonar sobre todos estos caminos exactamente, necesitamos el álgebra de caminos `kQ`.